In [4]:
import os, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, OrderedDict
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device('xpu' if torch.xpu.is_available() else 'cpu')

D_MODEL, NHEAD, NUM_LAYERS, FFN_DIM, DROPOUT = 128, 4, 2, 256, 0.1
BATCH_SIZE, MAX_EPOCHS, PATIENCE, LR = 256, 50, 5, 1e-3
MAX_SEQ_LEN = 20       # max complaint tokens per month (CLS prepended separately)
TSNE_BG     = 2000

SAVE            = True
PATH_SAVE_MODEL = '../data/02_artifacts/model_v2.pt'
PATH_LOAD_MODEL = '../data/02_artifacts/model_v2.pt'
print(f'device: {device}')

device: xpu


In [5]:
RAW_PATH = '../data/01_raw/311_180days.parquet'
os.makedirs('../data/01_raw', exist_ok=True)

if os.path.exists(RAW_PATH):
    df = pd.read_parquet(RAW_PATH)
    print(f'Loaded {len(df):,} records from cache')
else:
    import requests_cache
    from datetime import datetime, timedelta, timezone
    session = requests_cache.CachedSession('../data/00_cache/311', expire_after=3600)
    end   = datetime.now(timezone.utc)
    start = end - timedelta(days=180)
    where = (
        f"created_date >= '{start.strftime('%Y-%m-%dT00:00:00')}'"
        f" and created_date <= '{end.strftime('%Y-%m-%dT23:59:59')}'"
    )
    url, frames = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json', []
    for offset in range(0, 2_000_000, 50000):
        r = session.get(url, params={'$where': where, '$order': 'created_date ASC',
                                     '$limit': '50000', '$offset': str(offset)}, timeout=120)
        r.raise_for_status()
        batch = r.json()
        if not batch: break
        frames.append(pd.DataFrame(batch))
        print(f'  fetched {offset + len(batch):,}')
        if len(batch) < 50000: break
    df = pd.concat(frames, ignore_index=True)
    df['created_date'] = pd.to_datetime(df['created_date'])
    for col in ['latitude', 'longitude']:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'location' in df.columns: df = df.drop(columns=['location'])
    df.to_parquet(RAW_PATH)
    print(f'Fetched and saved {len(df):,} records')

  fetched 50,000
  fetched 100,000
  fetched 150,000
  fetched 200,000
  fetched 250,000
  fetched 300,000
  fetched 350,000
  fetched 400,000
  fetched 450,000
  fetched 500,000
  fetched 550,000
  fetched 600,000
  fetched 650,000
  fetched 700,000
  fetched 750,000
  fetched 800,000
  fetched 850,000
  fetched 900,000
  fetched 950,000
  fetched 1,000,000
  fetched 1,050,000
  fetched 1,100,000
  fetched 1,150,000
  fetched 1,200,000
  fetched 1,250,000
  fetched 1,300,000
  fetched 1,350,000
  fetched 1,400,000
  fetched 1,450,000
  fetched 1,500,000
  fetched 1,550,000
  fetched 1,600,000
  fetched 1,650,000
  fetched 1,700,000
  fetched 1,750,000
  fetched 1,800,000
  fetched 1,850,000
  fetched 1,900,000
  fetched 1,950,000
  fetched 1,975,929
Fetched and saved 1,975,929 records


In [6]:
df_clean = df.dropna(subset=['incident_address', 'complaint_type']).copy()
df_clean['created_date'] = pd.to_datetime(df_clean['created_date'])
df_clean['year_month']   = df_clean['created_date'].dt.to_period('M')

# ── TIER MAPPING ──────────────────────────────────────────────────────────────
# Edit complaint_type strings here to change grouping.
# Types not present in any list map to '[UNK]'.
TIER_MAP = OrderedDict([
    # Tier 1 — Life Safety
    ('T1_HVAC',      ['HEAT/HOT WATER', 'Boilers', 'Non-Residential Heat']),
    ('T1_ELECTRIC',  ['ELECTRIC', 'Electrical']),
    ('T1_WATER',     ['WATER LEAK', 'Water System']),
    ('T1_HAZMAT',    ['Lead', 'Asbestos', 'Hazardous Materials', 'Construction Lead Dust',
                      'Air Quality', 'Indoor Air Quality', 'Water Quality']),
    ('T1_SAFETY',    ['SAFETY', 'Emergency Response Team (ERT)', 'Scaffold Safety',
                      'Cranes and Derricks', 'BEST/Site Safety']),
    # Tier 2 — Housing Code
    ('T2_PLUMBING',  ['PLUMBING', 'Sewer', 'Root/Sewer/Sidewalk Condition', 'Indoor Sewage']),
    ('T2_STRUCTURE', ['DOOR/WINDOW', 'FLOORING/STAIRS', 'OUTSIDE BUILDING', 'PAINT/PLASTER']),
    ('T2_SANITATION',['UNSANITARY CONDITION', 'Rodent', 'Mold',
                      'Unsanitary Animal Pvt Property', 'Unsanitary Pigeon Condition']),
    ('T2_BUILDING',  ['Building/Use', 'General Construction/Plumbing', 'GENERAL', 'APPLIANCE',
                      'Elevator', 'ELEVATOR', 'Real Time Enforcement',
                      'Special Projects Inspection Team (SPIT)']),
    ('T2_FOOD',      ['Food Establishment', 'Food Poisoning', 'Day Care']),
    # Tier 3 — Quality of Life
    ('T3_NOISE',     ['Noise - Residential', 'Noise - Street/Sidewalk', 'Noise - Commercial',
                      'Noise - Vehicle', 'Noise', 'Noise - Helicopter',
                      'Noise - Park', 'Noise - House of Worship']),
    ('T3_DIRTY',     ['Dirty Condition', 'Illegal Dumping', 'Graffiti', 'Dead Animal',
                      'Litter Basket Complaint', 'Litter Basket Request']),
    ('T3_HOMELESS',  ['Encampment', 'Homeless Person Assistance']),
    ('T3_SOCIAL',    ['Drug Activity', 'Smoking or Vaping', 'Panhandling',
                      'Urinating in Public', 'Drinking', 'Non-Emergency Police Matter',
                      'Illegal Fireworks']),
    ('T3_WASTE',     ['Missed Collection', 'Residential Disposal Complaint',
                      'Commercial Disposal Complaint', 'Sanitation Worker or Vehicle Complaint']),
    # Tier 4 — Public Space
    ('T4_ROAD',      ['Street Condition', 'Sidewalk Condition', 'Curb Condition']),
    ('T4_PARKING',   ['Illegal Parking', 'Blocked Driveway', 'Abandoned Vehicle',
                      'Derelict Vehicles']),
    ('T4_TRAFFIC',   ['Traffic', 'Street Light Condition', 'Street Sign - Missing',
                      'Street Sign - Damaged', 'Street Sign - Dangling', 'Obstruction']),
    ('T4_TREES',     ['Damaged Tree', 'Dead/Dying Tree', 'Overgrown Tree/Branches',
                      'Illegal Tree Damage', 'New Tree Request', 'Uprooted Stump']),
    ('T4_OTHER',     ['Vendor Enforcement', 'Illegal Posting', 'For Hire Vehicle Complaint',
                      'Taxi Complaint', 'Taxi Report', 'For Hire Vehicle Report',
                      'Consumer Complaint', 'Animal in a Park', 'Violation of Park Rules',
                      'Lost Property', 'Outdoor Dining', 'Mobile Food Vendor',
                      'Water Conservation', 'Maintenance or Facility', 'Cannabis Retailer',
                      'Street Sweeping Complaint', 'Abandoned Bike', 'Bike/Roller/Skate',
                      'Animal-Abuse', 'Illegal Animal Kept as Pet', 'Unleashed Dog']),
])
CT_TO_TOKEN = {ct: tok for tok, cts in TIER_MAP.items() for ct in cts}
# ─────────────────────────────────────────────────────────────────────────────

df_clean['token'] = df_clean['complaint_type'].map(CT_TO_TOKEN).fillna('[UNK]')

all_months = pd.period_range(
    df_clean['created_date'].min().to_period('M'),
    df_clean['created_date'].max().to_period('M'), freq='M',
)
month_strs = [str(m) for m in all_months]
print(f'Months: {month_strs}')

monthly = (
    df_clean.sort_values('created_date')
    .groupby(['incident_address', 'year_month'])['token']
    .apply(lambda x: list(x)[:MAX_SEQ_LEN])
    .reset_index()
)
monthly.columns = ['address', 'month', 'tokens']
monthly['month'] = monthly['month'].astype(str)

addresses  = df_clean['incident_address'].unique()
full_idx   = pd.MultiIndex.from_product([addresses, month_strs], names=['address', 'month'])
monthly_full = (
    pd.DataFrame(index=full_idx).reset_index()
    .merge(monthly, on=['address', 'month'], how='left')
)
monthly_full['tokens'] = monthly_full['tokens'].apply(
    lambda x: x if isinstance(x, list) else ['NO_CALLS']
)
print(f'{len(monthly_full):,} (address, month) samples  |  {len(addresses):,} unique addresses')
unk_frac = (df_clean['token'] == '[UNK]').mean()
print(f'[UNK] fraction: {unk_frac:.1%}')

Months: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']
2,922,010 (address, month) samples  |  417,430 unique addresses
[UNK] fraction: 3.5%


In [7]:
# Vocab is defined by the tier map — no frequency counting needed
SPECIAL     = ['[PAD]', '[CLS]', '[UNK]', 'NO_CALLS']
TIER_TOKENS = list(TIER_MAP.keys())
vocab       = {t: i for i, t in enumerate(SPECIAL + TIER_TOKENS)}
inv_vocab   = {v: k for k, v in vocab.items()}
print(f'Vocab: {len(vocab)} tokens  ({len(TIER_TOKENS)} tier + {len(SPECIAL)} special)')

# Address-stratified 80/10/10 split
unique_addrs = monthly_full['address'].unique().tolist()
train_a, temp_a = train_test_split(unique_addrs, test_size=0.2, random_state=42)
val_a, test_a   = train_test_split(temp_a,       test_size=0.5, random_state=42)
split_map = {**{a: 'train' for a in train_a},
             **{a: 'val'   for a in val_a},
             **{a: 'test'  for a in test_a}}
monthly_full['split'] = monthly_full['address'].map(split_map)

# Consecutive (month_N, month_N+1) pairs — generalises to any number of months
pair_frames = []
for i in range(len(month_strs) - 1):
    mn, mn1 = month_strs[i], month_strs[i + 1]
    left  = (monthly_full[monthly_full['month'] == mn]
             [['address', 'split', 'month', 'tokens']]
             .rename(columns={'month': 'month_n', 'tokens': 'tokens_n'}))
    right = (monthly_full[monthly_full['month'] == mn1]
             [['address', 'tokens']]
             .rename(columns={'tokens': 'tokens_n1'}))
    pair_frames.append(left.merge(right, on='address').assign(month_n1=mn1))
pairs_df = pd.concat(pair_frames, ignore_index=True)

train_pairs = pairs_df[pairs_df['split'] == 'train']
val_pairs   = pairs_df[pairs_df['split'] == 'val']
test_pairs  = pairs_df[pairs_df['split'] == 'test']
print(f'Pairs — train:{len(train_pairs):,}  val:{len(val_pairs):,}  test:{len(test_pairs):,}')

Vocab: 24 tokens  (20 tier + 4 special)
Pairs — train:2,003,664  val:250,458  test:250,458


In [8]:
class NextMonthDataset(Dataset):
    def __init__(self, pairs, vocab, max_len=MAX_SEQ_LEN):
        self.pairs   = pairs.to_dict('records')
        self.vocab   = vocab
        self.max_len = max_len
        self.pad_id  = vocab['[PAD]']
        self.cls_id  = vocab['[CLS]']
        self.unk_id  = vocab['[UNK]']

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p      = self.pairs[idx]
        # Input: [CLS] + month_N tokens, padded to max_len + 1
        toks_n = p['tokens_n'][:self.max_len]
        ids    = [self.cls_id] + [self.vocab.get(t, self.unk_id) for t in toks_n]
        sl     = len(ids)
        attn   = [1] * sl + [0] * (self.max_len + 1 - sl)
        ids    = ids      + [self.pad_id] * (self.max_len + 1 - sl)
        # Target: soft frequency distribution over vocab for month_N+1
        target = torch.zeros(len(self.vocab))
        for t in p['tokens_n1']:
            target[self.vocab.get(t, self.unk_id)] += 1
        if target.sum() > 0:
            target /= target.sum()
        return {
            'input_ids':      torch.tensor(ids,  dtype=torch.long),
            'attention_mask': torch.tensor(attn, dtype=torch.long),
            'target':  target,
            'address': p['address'],
            'month':   p['month_n'],
        }


class AddressTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=NUM_LAYERS, ffn_dim=FFN_DIM, dropout=DROPOUT):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder   = nn.TransformerEncoder(layer, num_layers=num_layers,
                                                enable_nested_tensor=False)
        self.pred_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask):
        x       = self.embed(input_ids)
        x       = self.encoder(x, src_key_padding_mask=(attention_mask == 0))
        cls_emb = x[:, 0, :]          # position 0 is always [CLS]
        return self.pred_head(cls_emb), cls_emb

    def encode(self, input_ids, attention_mask):
        with torch.no_grad():
            x = self.embed(input_ids)
            x = self.encoder(x, src_key_padding_mask=(attention_mask == 0))
            return x[:, 0, :]         # CLS embedding

In [9]:
def collate_fn(batch):
    out = {}
    for k in batch[0]:
        out[k] = torch.stack([b[k] for b in batch]) if isinstance(batch[0][k], torch.Tensor)                  else [b[k] for b in batch]
    return out

train_ds = NextMonthDataset(train_pairs, vocab)
val_ds   = NextMonthDataset(val_pairs,   vocab)
test_ds  = NextMonthDataset(test_pairs,  vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

model     = AddressTransformer(len(vocab)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
print(f'Model params:   {sum(p.numel() for p in model.parameters()):,}')
print(f'Train batches:  {len(train_loader):,}  Val batches: {len(val_loader):,}')


def soft_ce(logits, target):
    return -(target * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()


def compute_mean_ll(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            target = batch['target'].to(device)
            logits, _ = model(ids, mask)
            ll     = (target * F.log_softmax(logits, dim=-1)).sum(dim=-1)
            total += ll.sum().item()
            n     += len(ll)
    return total / max(n, 1)

Model params:   271,128
Train batches:  7,827  Val batches: 979


In [10]:
if PATH_LOAD_MODEL is not None:
    model.load_state_dict(torch.load(PATH_LOAD_MODEL, map_location=device))
    print(f'Loaded model from {PATH_LOAD_MODEL}')
else:
    best_ll, best_state, no_improve = -float('inf'), None, 0
    history = []

    for epoch in range(MAX_EPOCHS):
        model.train()
        t_loss, n_batch = 0.0, 0
        for batch in train_loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            target = batch['target'].to(device)
            logits, _ = model(ids, mask)
            loss   = soft_ce(logits, target)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            t_loss  += loss.item()
            n_batch += 1

        train_loss = t_loss / n_batch
        val_ll     = compute_mean_ll(model, val_loader)
        history.append((train_loss, val_ll))
        print(f'epoch {epoch+1:3d}  train_loss={train_loss:.4f}  val_ll={val_ll:.4f}')

        if val_ll > best_ll:
            best_ll    = val_ll
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  early stop at epoch {epoch+1}  best_val_ll={best_ll:.4f}')
                break

    model.load_state_dict(best_state)

    if SAVE:
        os.makedirs(os.path.dirname(PATH_SAVE_MODEL), exist_ok=True)
        torch.save(model.state_dict(), PATH_SAVE_MODEL)
        print(f'Saved to {PATH_SAVE_MODEL}')

    train_losses, val_lls = zip(*history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses); axes[0].set_title('Train loss (soft-CE)'); axes[0].set_xlabel('epoch')
    axes[1].plot(val_lls);      axes[1].set_title('Val log-likelihood');   axes[1].set_xlabel('epoch')
    plt.tight_layout(); plt.show()

Loaded model from ../data/02_artifacts/model_v2.pt


In [11]:
test_ll     = compute_mean_ll(model, test_loader)
baseline_ll = -math.log(len(vocab))   # uniform over all tokens

print(f'Test mean log-likelihood:  {test_ll:.4f} nats')
print(f'Baseline (uniform):        {baseline_ll:.4f} nats')
print(f'Improvement over baseline: {test_ll - baseline_ll:+.4f} nats')

Test mean log-likelihood:  -4.1907 nats
Baseline (uniform):        -3.1781 nats
Improvement over baseline: -1.0127 nats


In [12]:
model.eval()
cls_id = vocab['[CLS]']
pad_id = vocab['[PAD]']
unk_id = vocab['[UNK]']

all_embs, all_meta = [], []
b_ids, b_masks, b_meta = [], [], []

def flush():
    if not b_ids: return
    ids_t  = torch.tensor(b_ids,   dtype=torch.long).to(device)
    mask_t = torch.tensor(b_masks, dtype=torch.long).to(device)
    all_embs.append(model.encode(ids_t, mask_t).cpu().numpy())
    all_meta.extend(b_meta)
    b_ids.clear(); b_masks.clear(); b_meta.clear()

for rec in monthly_full.to_dict('records'):
    toks    = rec['tokens'][:MAX_SEQ_LEN]
    ids     = [cls_id] + [vocab.get(t, unk_id) for t in toks]
    sl      = len(ids)
    attn    = [1] * sl + [0] * (MAX_SEQ_LEN + 1 - sl)
    ids     = ids + [pad_id] * (MAX_SEQ_LEN + 1 - sl)
    b_ids.append(ids); b_masks.append(attn)
    b_meta.append({'address': rec['address'], 'month': rec['month'], 'split': rec['split']})
    if len(b_ids) >= BATCH_SIZE: flush()
flush()

all_embs = np.vstack(all_embs)
emb_df   = pd.DataFrame(all_meta)

def classify_tier(toks):
    if toks == ['NO_CALLS']: return 'no_calls'
    if any(t.startswith(('T1', 'T2')) for t in toks): return 'T1T2'
    return 'T3T4'

monthly_full['tier'] = monthly_full['tokens'].apply(classify_tier)
emb_df = emb_df.merge(monthly_full[['address', 'month', 'tier']], on=['address', 'month'])
print(f'Embeddings: {all_embs.shape}')
print(emb_df['tier'].value_counts().to_string())

Embeddings: (2922010, 128)
tier
no_calls    2193196
T3T4         540344
T1T2         188470


In [13]:
# ── compute per-pair log-likelihoods (test set) ──────────────────────────────
model.eval()
ll_records = []
with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        target = batch['target'].to(device)
        logits, _ = model(ids, mask)
        ll = (target * F.log_softmax(logits, dim=-1)).sum(dim=-1).cpu().numpy()
        for i in range(len(ll)):
            ll_records.append({'address': batch['address'][i],
                               'month_n': batch['month'][i],
                               'll': float(ll[i])})
ll_df     = pd.DataFrame(ll_records)
ll_lookup = dict(zip(zip(ll_df['address'], ll_df['month_n']), ll_df['ll']))

tok_lookup = monthly_full.set_index(['address', 'month'])['tokens']

# ── TSNE on month[0] CLS embeddings, colored by P(month[1] | month[0]) ───────
src_month = month_strs[0]
tgt_month = month_strs[1]

src_mask = (emb_df['month'] == src_month) & (emb_df['split'] == 'test')
src_embs = all_embs[src_mask.values]
src_meta = emb_df[src_mask].reset_index(drop=True)

rng_idx = np.random.choice(len(src_embs),
                            size=min(TSNE_BG + 500, len(src_embs)), replace=False)
coords  = TSNE(n_components=2, random_state=42,
               perplexity=min(30, len(rng_idx) - 1)).fit_transform(src_embs[rng_idx])
sub     = src_meta.iloc[rng_idx].reset_index(drop=True)

def fmt_toks(toks):
    c = Counter(toks)
    return '  '.join(f'{t}×{n}' if n > 1 else t for t, n in c.most_common())

ll_vals = np.array([ll_lookup.get((r['address'], src_month), float('nan'))
                    for _, r in sub.iterrows()])
hover = [
    f'<b>{r["address"]}</b><br>'
    f'll: {ll_lookup.get((r["address"], src_month), float("nan")):.3f}<br>'
    f'{src_month}: {fmt_toks(tok_lookup.get((r["address"], src_month), ["NO_CALLS"]))}<br>'
    f'→ {tgt_month}: {fmt_toks(tok_lookup.get((r["address"], tgt_month), ["NO_CALLS"]))}'
    for _, r in sub.iterrows()
]

valid = ~np.isnan(ll_vals)
fig   = go.Figure()
if (~valid).any():
    fig.add_trace(go.Scatter(
        x=coords[~valid, 0], y=coords[~valid, 1],
        mode='markers', marker=dict(color='#e5e7eb', size=4, opacity=0.3),
        hoverinfo='skip', showlegend=False,
    ))
fig.add_trace(go.Scatter(
    x=coords[valid, 0], y=coords[valid, 1],
    mode='markers', name='test pairs',
    marker=dict(
        color=ll_vals[valid],
        colorscale='RdBu',
        reversescale=False,    # red = low LL (surprising), blue = high LL (expected)
        size=6, opacity=0.85,
        colorbar=dict(title='log-likelihood', thickness=15, len=0.75),
        showscale=True,
    ),
    text=np.array(hover)[valid],
    hoverinfo='text',
))
fig.update_layout(
    title=(f'TSNE – {src_month} CLS embeddings  '
           f'| color = P({tgt_month} | {src_month})  (red = surprising)'),
    xaxis_title='TSNE-1', yaxis_title='TSNE-2',
    width=950, height=720,
)
fig.show()

In [14]:
# ll_df and tok_lookup are set in tsne-month
T1T2_TOKENS = {t for t in TIER_MAP if t.startswith(('T1', 'T2'))}

def next_group(addr, month_n):
    i    = month_strs.index(month_n)
    toks = tok_lookup.get((addr, month_strs[i + 1]), ['NO_CALLS'])
    if toks == ['NO_CALLS']: return 'NO_CALLS next'
    frac = sum(1 for t in toks if t in T1T2_TOKENS) / len(toks)
    return 'T1+T2 majority next' if frac >= 0.5 else 'T3+T4 majority next'

ll_df['group'] = [next_group(r['address'], r['month_n']) for _, r in ll_df.iterrows()]

GROUP_COLORS = {
    'T1+T2 majority next': '#ef4444',
    'T3+T4 majority next': '#6b7280',
    'NO_CALLS next':       '#d1d5db',
}

fig = go.Figure()
for group, color in GROUP_COLORS.items():
    sub = ll_df[ll_df['group'] == group]
    if sub.empty: continue
    fig.add_trace(go.Histogram(
        x=sub['ll'], name=f'{group}  (n={len(sub):,})',
        opacity=0.7, marker_color=color,
        xbins=dict(size=0.05),
    ))

baseline = -math.log(len(vocab))
fig.add_vline(x=baseline, line_dash='dash', line_color='black',
              annotation_text='uniform baseline', annotation_position='top right')
fig.update_layout(
    barmode='overlay',
    title='Log-likelihood of next month | current month  (test set)',
    xaxis_title='log-likelihood (nats)',
    yaxis_title='count',
    width=950, height=600,
    legend=dict(font=dict(size=11)),
)
fig.show()

print(f'{"Group":<25}  {"n":>6}  {"mean_ll":>8}  {"median_ll":>10}')
print('-' * 55)
for group in GROUP_COLORS:
    sub = ll_df[ll_df['group'] == group]
    if sub.empty: continue
    print(f'{group:<25}  {len(sub):>6,}  {sub["ll"].mean():>8.3f}  {sub["ll"].median():>10.3f}')

Group                           n   mean_ll   median_ll
-------------------------------------------------------
T1+T2 majority next        13,903    -3.475      -3.379
T3+T4 majority next        47,271    -2.435      -1.820
NO_CALLS next              189,284    -4.682      -6.293
